# 03. Vetorização Clássica (BoW, N-grams, TF-IDF) e Baseline de ML

**Contexto:** Tech Challenge 04 - Voice of Customer Intelligence com RAG  
**Fundamentação Teórica:** Fase 4 | Aula 03 - *Vetorizando Texto: Bag of Words e TF-IDF*  

Antes de partir para modelos neurais pesados e bancos vetoriais densos, um cientista de dados sênior deve dominar a **vetorização baseada em frequência** e estabelecer **baselines clássicos e interpretáveis**. Como visto na Aula 03, em 2026 o TF-IDF (e variantes como BM25) ainda é a coluna vertebral dos sistemas de Information Retrieval e serve como primeiro estágio em pipelines de **Busca Híbrida em RAG**.

### Objetivos deste Notebook:
1. Implementar o **Bag of Words (BoW)** e mensurar a esparsidade da matriz documento-termo.
2. Aplicar **N-grams (1, 2)** para capturar o contexto local de negações em português (*"não recebi"*, *"veio quebrado"*).
3. Aplicar **TF-IDF** com sublinear scaling e extrair as palavras/expressões mais discriminativas para notas baixas (problemas) vs notas altas (elogios).
4. Treinar e avaliar um **Baseline de Machine Learning** (Regressão Logística com Validação Cruzada 5-fold e F1-score) comparando BoW vs TF-IDF.
5. Fundamentar tecnicamente a transição para os **Embeddings Densos e ChromaDB** da próxima fase.

In [ ]:
import sys
import os
sys.path.append('../')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score
from sklearn.metrics import classification_report

from src.vectorization import (
    criar_bow_vectorizer,
    criar_tfidf_vectorizer,
    calcular_metricas_matriz_esparsa,
    extrair_top_features_por_classe,
)

DATA_DIR = '../data/processed/'
df_processed = pd.read_csv(f"{DATA_DIR}reviews_enriched_for_rag.csv")
print(f"Total de avaliações processadas carregadas: {len(df_processed):,}")

## 1. Bag of Words (BoW): A Primeira Vetorização (Aula 03)
O BoW mapeia cada termo do vocabulário para uma coluna e conta as ocorrências. Vamos analisar a dimensionalidade e a esparsidade geradas no corpus.

In [ ]:
corpus_textos = df_processed['clean_comment'].fillna('').astype(str).tolist()

bow_vec = CountVectorizer(lowercase=True, min_df=5, max_features=5000)
X_bow = bow_vec.fit_transform(corpus_textos)

metricas_bow = calcular_metricas_matriz_esparsa(X_bow)
print("=== Métricas da Matriz Documento-Termo (BoW) ===")
for k, v in metricas_bow.items():
    print(f"{k}: {v:,.2f}" if isinstance(v, float) else f"{k}: {v:,}")

## 2. N-grams: Capturando Contexto Local e Negações (Aula 03)
Como visto na Aula 03, o BoW puro tem uma falha grave: para ele, *"não gostei"* e *"gostei"* são quase iguais porque ambos compartilham o termo *"gostei"*.  
Ao usarmos `ngram_range=(1, 2)`, geramos **bigrams** como *"não recebi"*, *"muito bom"* e *"veio quebrado"*, capturando modificadores e negações locais.

In [ ]:
bigram_vec = CountVectorizer(ngram_range=(2, 2), min_df=10, max_features=20)
X_bigram = bigram_vec.fit_transform(corpus_textos)
top_bigrams = bigram_vec.get_feature_names_out()

print("Exemplos de Bigrams mais frequentes capturando contexto de compra:")
for bg in top_bigrams:
    print(f"- '{bg}'")

## 3. TF-IDF e Termos Mais Discriminativos por Sentimento (Aula 03)
O TF-IDF (Term Frequency - Inverse Document Frequency) atenua palavras universais que aparecem em quase todos os documentos e realça termos altamente específicos.  
Vamos segregar avaliações negativas (Notas 1 e 2) vs positivas (Notas 4 e 5) e descobrir os termos de maior peso TF-IDF para cada uma.

In [ ]:
# Rotulação binária para diagnóstico de sentimento
# 0 = Negativo (Notas 1 e 2) | 1 = Positivo (Notas 4 e 5)
df_polar = df_processed[df_processed['review_score'] != 3].copy()
df_polar['target_sentimento'] = (df_polar['review_score'] >= 4).astype(int)

tfidf_vec = criar_tfidf_vectorizer(
    ngram_range=(1, 2),
    min_df=5,
    max_df=0.85,
    max_features=8000,
    sublinear_tf=True,
)

X_tfidf = tfidf_vec.fit_transform(df_polar['clean_comment'].fillna(''))
y_polar = df_polar['target_sentimento'].values
features = tfidf_vec.get_feature_names_out()

top_features = extrair_top_features_por_classe(X_tfidf, y_polar, features, top_n=10)

print("=== TOP 10 TERMOS DISCRIMINATIVOS - NEGATIVOS (Notas 1-2) ===")
for termo, peso in top_features['0']:
    print(f"  {termo:<25} (peso médio TF-IDF: {peso:.4f})")

print("\n=== TOP 10 TERMOS DISCRIMINATIVOS - POSITIVOS (Notas 4-5) ===")
for termo, peso in top_features['1']:
    print(f"  {termo:<25} (peso médio TF-IDF: {peso:.4f})")

## 4. Comparação Experimental: Baseline de ML (BoW vs TF-IDF) (Aula 03)
Seguindo o roteiro do Passo 3 da Aula 03, treinamos uma Regressão Logística para classificar o sentimento do cliente usando validação cruzada 5-fold com a métrica **F1-score**.

In [ ]:
# Subsample estratificado balanceado de 5.000 amostras para validação rápida e reproduzível
df_sample = df_polar.groupby('target_sentimento', group_keys=False).apply(lambda x: x.sample(2500, random_state=42))
textos_sample = df_sample['clean_comment'].fillna('')
y_sample = df_sample['target_sentimento'].values

# 1) Pipeline BoW
bow_exp = CountVectorizer(ngram_range=(1, 2), min_df=3, max_features=5000)
X_bow_sample = bow_exp.fit_transform(textos_sample)
score_bow = cross_val_score(LogisticRegression(max_iter=500), X_bow_sample, y_sample, cv=5, scoring='f1').mean()

# 2) Pipeline TF-IDF
tfidf_exp = TfidfVectorizer(ngram_range=(1, 2), min_df=3, max_features=5000, sublinear_tf=True)
X_tfidf_sample = tfidf_exp.fit_transform(textos_sample)
score_tfidf = cross_val_score(LogisticRegression(max_iter=500), X_tfidf_sample, y_sample, cv=5, scoring='f1').mean()

print("=== Resultado da Avaliação Comparativa (5-Fold Cross Validation) ===")
print(f"BoW F1-Score:    {score_bow:.4f} ({score_bow*100:.2f}%)")
print(f"TF-IDF F1-Score: {score_tfidf:.4f} ({score_tfidf*100:.2f}%)")
print(f"Ganho do TF-IDF: {(score_tfidf - score_bow)*100:+.2f} pontos percentuais de F1.")

## 5. Conclusões e Conexão com o RAG (Fase 4 | Aula 03)

1. **Poder Discriminativo:** O TF-IDF isolou com maestria os padrões das reclamações (*"não recebi"*, *"atraso"*, *"errado"*, *"quebrado"*) e dos elogios (*"chegou antes"*, *"recomendo"*, *"ótimo"*).
2. **Limitação da Vetorização de Frequência:** Apesar de eficiente para classificação binária, o TF-IDF gera vetores ortogonais esparsos que não capturam sinônimos abstratos nem permitem perguntas abertas em linguagem natural.
3. **Transição para RAG:** Nos próximos notebooks (`04_banco_vetorial_chroma.ipynb` e `05_pipeline_rag_llm.ipynb`), passamos da frequência para **Embeddings Semânticos Densos** no **ChromaDB**, onde perguntas complexas de gestores serão respondidas com precisão e contextualização enriquecida.